In [ ]:
# =========================
# Cell 1 - Install library
# =========================
%pip install -q pandas numpy scikit-learn nltk matplotlib seaborn


# =========================
# Cell 2 - Import & Load Data
# =========================
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Pastikan file CSV sudah ada di folder kerja dengan nama berikut:
CSV_PATH = "IMDB Dataset.csv"
df = pd.read_csv(CSV_PATH)

print("Shape data:", df.shape)
print(df.head())


# =========================
# Cell 3 - Preprocessing Text (Ekstraksi Fitur Dasar)
# =========================
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)          # hapus HTML tag
    text = re.sub(r"[^a-z\s]", " ", text)       # hanya huruf
    text = re.sub(r"\s+", " ", text).strip()    # rapikan spasi
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return " ".join(tokens)

df["clean_review"] = df["review"].apply(clean_text)
print(df[["review", "clean_review"]].head(3))


# =========================
# Cell 4 - Tabel Term & Jumlah Kemunculan di CSV
# =========================
# Count semua term dari seluruh dokumen
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(df["clean_review"])

terms = count_vectorizer.get_feature_names_out()
term_counts = np.asarray(X_count.sum(axis=0)).ravel()

# Format long (mudah dibaca): term | jumlah
term_freq_long = pd.DataFrame({
    "term": terms,
    "jumlah_kemunculan": term_counts
}).sort_values("jumlah_kemunculan", ascending=False).reset_index(drop=True)

# Format wide (sesuai permintaan: kolom = term, row = jumlah)
term_freq_wide = pd.DataFrame([term_counts], columns=terms, index=["jumlah_kemunculan"])

# Simpan
term_freq_long.to_csv("term_frequency_long.csv", index=False)
term_freq_wide.to_csv("term_frequency_wide.csv", index=True)

print("Top 20 term paling sering muncul:")
print(term_freq_long.head(20))
print("\nUkuran tabel wide:", term_freq_wide.shape)


# =========================
# Cell 5 - TF-IDF Matrix
# =========================
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)
X_tfidf = tfidf_vectorizer.fit_transform(df["clean_review"])
vocab = tfidf_vectorizer.get_feature_names_out()

print("Shape TF-IDF:", X_tfidf.shape)


# =========================
# Cell 6 - Clustering (KMeans, k=2) pada matriks TF-IDF 8 x vocabulary
# =========================
# Ambil 8 dokumen pertama -> matriks 8 x vocabulary
X_8 = X_tfidf[:8]
df_8 = df.iloc[:8].copy()

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_8)
df_8["cluster"] = clusters

# Output cluster 0 dan 1
print("Hasil cluster untuk 8 dokumen pertama:")
print(df_8[["sentiment", "cluster"]])

# Interpretasi cluster: ambil top term dari centroid
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
top_n = 10
for i in range(2):
    top_terms = [vocab[idx] for idx in order_centroids[i, :top_n]]
    print(f"\nCluster {i} top terms:", ", ".join(top_terms))

# Evaluasi clustering
# (Silhouette dkk valid bila tiap cluster berisi setidaknya 2 sampel total dan >1 cluster terbentuk)
unique_clusters = np.unique(clusters)
if len(unique_clusters) > 1 and len(df_8) > len(unique_clusters):
    sil = silhouette_score(X_8, clusters)
    dbi = davies_bouldin_score(X_8.toarray(), clusters)
    chi = calinski_harabasz_score(X_8.toarray(), clusters)
    print("\nEvaluasi Clustering:")
    print("Silhouette Score      :", round(sil, 4))
    print("Davies-Bouldin Index  :", round(dbi, 4))
    print("Calinski-Harabasz     :", round(chi, 4))
else:
    print("\nEvaluasi clustering tidak bisa dihitung karena distribusi cluster tidak valid.")


# =========================
# Cell 7 - Classification (TF-IDF + Logistic Regression)
# =========================
# Label: positive=1, negative=0
y = df["sentiment"].map({"negative": 0, "positive": 1}).values

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Akurasi:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))